<a href="https://colab.research.google.com/github/BrunoAG77/simulacao-llm-opiniao/blob/main/llmia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#**Percepção dos brasileiros sobre o racismo no Brasil**
####Integrantes do Grupo
#####10417318, Bruno Antico Galin
#####10391119, Cleverson Pereira da Silva
#####10417877, Eduardo Takashi Missaka
#####10418552, Ismael de Sousa Silva
#####10410862, Vitor Alves Pereira


---
##Parte 1 - Inicialização e leitura

In [ ]:
#
!pip install pyreadstat

In [4]:
import pandas as pd
import pyreadstat
import torch
import re
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, precision_score, recall_score, f1_score
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from transformers import AutoTokenizer, AutoModelForCausalLM

In [5]:
df = pd.read_spss('04829.SAV') #Leitura do arquivo

In [ ]:
print(df.columns)

####**P01) (CARTELA 1) Dentre as questões listadas nessa cartela, na sua opinião, qual é a que mais contribui para gerar as desigualdades no Brasil? (RU)**
#####01( ) Raça/Cor/Etnia
#####02( ) Classe social
#####03( ) Gênero ou sexo
#####04( ) Local de moradia (cidade, região da cidade, bairro, comunidade)
#####05( ) Deficiência (auditiva, visual, intelectual e motora)
#####06( ) Orientação sexual (heterossexual, homossexual)
#####07( ) Local de origem/onde nasceu (país ou região do país)
#####97( ) Nenhuma destas/Outras (sem especificar)
#####98( ) Não sabe
#####99( ) Não respondeu

In [ ]:
df['P1'].value_counts()

In [ ]:
df['P1'].replace({
    'Raça/Cor/Etnia':1,
    'Classe social':2,
    'Gênero ou sexo':3,
    'Local de moradia (cidade, região da cidade, bairro, comunidade)':4,
    'Deficiência (auditiva, visual, intelectual e motora)':5,
    'Orientação sexual (heterossexual, homossexual)':6,
    'Local de origem/onde nasceu (país ou região do país)':7,
    'Nenhuma destas/Outras (sem especificar)':97,
    'Não sabe/ Não respondeu':9899,
    }, inplace=True) #Codificação das respostas de P1

####**P02) Agora gostaria de saber como o(a) sr(a) se sente ao ter que responder qual sua raça/cor/etnia. Você diria que se sente: (RU – LEIA ALTERNATIVAS, NÃO LEIA “NÃO SABE” E “NÃO RESPONDEU)**
#####01( ) Muito confortável
#####02( ) Confortável
#####03( ) Desconfortável
#####04( ) Muito desconfortável
#####98( ) Não sabe
#####99( ) Não respondeu

In [ ]:
df['P2'].value_counts()

In [ ]:
df['P2'].replace({
    'Muito confortável':1,
    'Confortável':2,
    'Desconfortável':3,
    'Muito desconfortável':4,
    'Não sabe/ Não respondeu':9899,
    }, inplace=True) #Codificação das respostas de P2

**P03) Para o(a) sr(a), definir a sua raça/cor/etnia é muito fácil, fácil, difícil ou muito difícil? (RU)**

#####01( ) Muito fácil
#####02( ) Fácil
#####03( ) Difícil
#####04( ) Muito difícil
#####98( ) Não sabe
#####99( ) Não respondeu

In [ ]:
df['P3'].value_counts()

In [ ]:
df['P3'].replace({
    'Muito fácil':1,
    'Fácil':2,
    'Difícil':3,
    'Muito difícil':4,
    'Não sabe/ Não respondeu':9899,
    }, inplace=True) #Codificação das respostas de P3

**P04) Na sua opinião, declarar a sua raça/cor/etnia é muito, pouco ou nada importante? (RU)**
#####01( ) Muito importante
#####02( ) Pouco importante
#####03( ) Nada importante
#####98( ) Não sabe
#####99( ) Não respondeu

In [ ]:
df['P4'].value_counts()

In [ ]:
df['P4'].replace({
    'Muito importante':1,
    'Pouco importante':2,
    'Nada importante':3,
    'Não sabe/ Não respondeu':9899,
    }, inplace=True) #Codificação das respostas de P4

In [15]:
perguntas = {
    "P1":{
        "texto": "Dentre as questões listadas nessa cartela, na sua opinião, qual é a que mais contribui para gerar as desigualdades no Brasil?",
        "opcoes": {
          1:'Raça/Cor/Etnia',
          2:'Classe social',
          3:'Gênero ou sexo',
          4:'Local de moradia (cidade, região da cidade, bairro, comunidade)',
          5:'Deficiência (auditiva, visual, intelectual e motora)',
          6:'Orientação sexual (heterossexual, homossexual)',
          7:'Local de origem/onde nasceu (país ou região do país)',
          97:'Nenhuma destas/Outras (sem especificar)',
        }
    },

        "P2":{
        "texto": "Agora gostaria de saber como o(a) sr(a) se sente ao ter que responder qual sua raça/cor/etnia. Você diria que se sente:",
        "opcoes": {
          1:'Muito confortável',
          2:'Confortável',
          3:'Desconfortável',
          4:'Muito desconfortável',
        }
    },

        "P3":{
        "texto": "Para o(a) sr(a), definir a sua raça/cor/etnia é muito fácil, fácil, difícil ou muito difícil?",
        "opcoes": {
          1:'Muito fácil',
          2:'Fácil',
          3:'Difícil',
          4:'Muito difícil',
        }
    },

        "P4":{
        "texto": "Na sua opinião, declarar a sua raça/cor/etnia é muito, pouco ou nada importante?",
        "opcoes": {
          1:'Muito importante',
          2:'Pouco importante',
          3:'Nada importante',
        }
    }
} #Base de perguntas para o LLM

In [24]:
targets = ['P1','P2','P3','P4'] #Lista de perguntas alvo para o LLM
features = ['SEXO','IDADE','RACA','ESCOLARIDADE','UF','MUNI','REGIAO','RELIGIAO'] #Features para o LLM
executar_llm = False #Variável booleana para executar o LLM ou não. Caso execute no ambiente de CPU, colocar igual a False se não o LLM entra em loop. Caso execute no ambiente de CPU, colocar igual a True para o LLM rodar e com um tempo rápido

---
##Parte 2 - Modelo LLM

In [25]:
dfmodel = df[features + targets] #Modelo do LLM, com features e perguntas alvo

In [26]:
def extrair_numero(resposta): #Função para extrair a resposta do DataFrame
    match = re.search(r'\d+', resposta)
    return int(match.group()) if match else None

In [27]:
def send_input(lin, pergunta): #Prompt para o LLM
  opcoes_txt = "\n".join([f"{k} {v}" for k, v in pergunta["opcoes"].items()])
  return f"""
Perfil:
Idade: {lin['IDADE']}
Sexo: {lin['SEXO']}
Raça: {lin['RACA']}

Pergunta:
{pergunta["texto"]}

Opções:
{opcoes_txt}

Responda apenas com um número:
"""

In [28]:
def gerar(prompts,batch_size=8): #Função para gerar prompt para o LLM
   respostas = []
   for i in range(0, len(prompts), batch_size):
        batch = prompts[i:i+batch_size] #Batch (Lotes) para mandar várias inferências por vez, para otimização de tempo

        device = "cuda" if torch.cuda.is_available() else "cpu" #Ambiente de execução (CPU ou GPU)

        inputs = tokenizer(batch, return_tensors="pt",padding=True,truncation=True,padding_side='left').to(device) #Tokenizador dos prompts
        model.to(device) #Move o modelo para o ambiente escolhido (CPU ou GPY)

        outputs = model.generate(
            **inputs,
            max_new_tokens=2,
            do_sample=True,
            temperature=0.3,
            top_p=0.8,
            pad_token_id=tokenizer.eos_token_id
        ) #Geração de resposta do modelo

        decoded = tokenizer.batch_decode(outputs, skip_special_tokens=True) #Conversão dos tokens gerados para texto

        for p, d in zip(batch, decoded): #Remove o prompt original da saída e mantém a resposta gerada
            respostas.append(d[len(p):].strip())
   return respostas

In [29]:
def executar(df, pergunta, nsamples=200): #Função de execução do LLM
  dfsample = df.sample(nsamples)
  prompts = [send_input(lin, pergunta) for lin in dfsample.to_dict("records")]
  respostas = gerar(prompts,batch_size=8)
  y_llm = [extrair_numero(r) for r in respostas]
  return dfsample, y_llm

In [30]:
metricasllm = [] #Métricas do LLM
if executar_llm: #Roda a função se executar_llm = True no ambiente de GPU conforme explicado antes
  model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0" #Modelo Llama
  tokenizer = AutoTokenizer.from_pretrained(model_name) #Tokenizador
  model = AutoModelForCausalLM.from_pretrained(model_name,device_map="auto") #torch_dtype=torch.float16

  for rep in range(3): #3 repetições
      print(f"\n=====Repetição {rep}=====")

      resultados = {}

      for id, pergunta in perguntas.items():
        dfsample, y_llm = executar(dfmodel, pergunta, nsamples=200) #200 amostras, 10% do DataFrame original
        resultados[id] = {"dfsample":dfsample,"y_llm":y_llm}

        y_real = resultados[id]["dfsample"][id].reset_index(drop=True) #Valores reais
        y_llm = pd.Series(resultados[id]["y_llm"]).reset_index(drop=True) #Previsões do LLM para séries Pandas

        opcoes_nonull = list(perguntas[id]["opcoes"].keys()) #Opções válidas

        y_llm = y_llm.apply(lambda x: x if x in opcoes_nonull else None) #Tratamento de respostas nulas

        mask = y_real.notnull() & y_llm.notnull() #Máscara para remover valores nulos

        y_real = y_real[mask]
        y_llm = y_llm[mask]

        y_real_percent = y_real.value_counts(normalize=True) #Distribuição percentual das classes reais
        y_llm_percent = y_llm.value_counts(normalize=True) #Distribuição percentual das classes previstas
        accuracy_metric = accuracy_score(y_real, y_llm) #Acurácia
        precision_metric = precision_score(y_real, y_llm, average="weighted") #Precisão
        recall_metric = recall_score(y_real, y_llm, average="weighted") #Recall
        f1_metric = f1_score(y_real, y_llm, average="weighted") #F1-Score
        report_metric = classification_report(y_real, y_llm) #Relatório de classificação

        metricasllm.append({
            "Repetição":rep+1,
            "Pergunta":id,
            "YReal":y_real_percent,
            "YLLM":y_llm_percent,
            "Acurácia":accuracy_metric,
            "Precisão":precision_metric,
            "Recall":recall_metric,
            "F1":f1_metric,
            "Report":report_metric
        })

        print("\nREAL:")
        print(y_real.value_counts(normalize=True))

        print("\nLLM:")
        print(y_llm.value_counts(normalize=True))

        print("\nAcurácia:")
        print(accuracy_score(y_real, y_llm))

        print("\nRelatório de classificação:")
        print(classification_report(y_real, y_llm))

      with open("resultados.pkl","wb") as f: #Salva os resultados da execução em um arquivo pickle, assim não precisa rodar a função quando executar_llm = False e em ambiente de CPU
        pickle.dump(resultados,f)

      with open("metricasllm.pkl","wb") as f: #Salva as métricas da execução em um arquivo pickle, assim não precisa rodar a função quando executar_llm = False e em ambiente de CPU
        pickle.dump(metricasllm,f)
else:
  with open("resultados.pkl","rb") as f: #Abre o arquivo pickle de resultados
    resultados = pickle.load(f)
  with open("metricasllm.pkl","rb") as f: #Abre o arquivo pickle de métricas
    metricasllm = pickle.load(f)

dfmetricas = pd.DataFrame(metricasllm)

In [31]:
dfmetricas = dfmetricas.groupby("Pergunta")[["Acurácia","Precisão","Recall","F1"]].mean().reset_index()

In [ ]:
sns.barplot(x=dfmetricas["Pergunta"],y=dfmetricas["Acurácia"],data=dfmetricas,color="red")
plt.title("Barplot - Acurácia do LLM por pergunta")
plt.show()

In [ ]:
sns.boxplot(data=dfmetricas[["Precisão","Recall","F1"]])
plt.title("Boxplot - Precisão, Recall e F1-Score do LLM por pergunta")
plt.ylabel("Percentual")
plt.show()

In [ ]:
metricas_calor = dfmetricas.set_index("Pergunta")[["Acurácia","Precisão","Recall","F1"]]

sns.heatmap(metricas_calor,annot=True)
plt.title("Mapa de Calor - Métricas do LLM por pergunta")
plt.show()

In [ ]:
plt.title("Comparação Distribuição Real X LLM por pergunta")
for id in resultados:
  dist_y_real = resultados[id]["dfsample"][id].value_counts(normalize=True).sort_index()
  dist_y_llm = pd.Series(resultados[id]["y_llm"]).value_counts(normalize=True).sort_index()

  dist = pd.DataFrame({
      "Pergunta":dist_y_real.index,
      "Real":dist_y_real.values,
      "LLM":dist_y_llm.reindex(dist_y_real.index,fill_value=0).values
    })

  longdist = dist.melt(id_vars="Pergunta",var_name="Origem",value_name="Distribuição")
  sns.barplot(x="Pergunta",y="Distribuição",data=longdist,hue="Origem",errorbar=None)
  plt.xlabel(f"{id}")
  plt.show()

---
##Parte 3 - Modelo de aprendizado de máquina supervisionado (RandomForest)

In [ ]:
rfmetricas = [] #Métricas do RandomForest

for t in targets:
  print(f"\n====={t}=====")

  dfrf = dfmodel.copy()
  dfrf[t] = pd.to_numeric(dfrf[t])
  dfrf = dfrf[dfrf[t] != 9899] #Tratamento de valores nulos
  dfrf[features] = dfrf[features].astype(str)
  dfrf[features] = dfrf[features].fillna("Desconhecido")
  dfrf = dfrf.dropna(subset=[t])

  x = dfrf[features] #Features
  y = dfrf[t] #Perguntas alvo

  x = pd.get_dummies(x) #Criação de colunas de codificação

  x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2,random_state=42) #Separação treino/teste

  rfmodel = RandomForestClassifier(n_estimators=100, random_state=42) #Criação do modelo RandomForest
  rfmodel.fit(x_train, y_train) #Treinamento do modelo

  y_pred = rfmodel.predict(x_test) #Previsão do modelo

  accuracy_metric = accuracy_score(y_test, y_pred)
  precision_metric = precision_score(y_test, y_pred, average="weighted")
  recall_metric = recall_score(y_test, y_pred, average="weighted")
  f1_metric = f1_score(y_test, y_pred, average="weighted")
  report_metric = classification_report(y_test, y_pred)

  rfmetricas.append({
        "Pergunta":t,
        "Acurácia":accuracy_metric,
        "Precisão":precision_metric,
        "Recall":recall_metric,
        "F1":f1_metric,
        "Report":report_metric
  })

  print("\nAcurácia: ")
  print(accuracy_score(y_test, y_pred))
  print("\nRelatório: ")
  print(classification_report(y_test, y_pred))

dfrfmetricas = pd.DataFrame(rfmetricas)

In [ ]:
sns.barplot(x=dfrfmetricas["Pergunta"],y=dfrfmetricas["Acurácia"],data=dfrfmetricas,color="red",errorbar=None)
plt.title("Barplot - Acurácia do RandomForest por pergunta")
plt.show()

In [ ]:
sns.boxplot(data=dfrfmetricas[["Precisão","Recall","F1"]])
plt.title("Boxplot - Precisão, Recall e F1-Score do RandomForest por pergunta")
plt.ylabel("Percentual")
plt.show()

In [ ]:
metricasrf_calor = dfrfmetricas.set_index("Pergunta")[["Acurácia","Precisão","Recall","F1"]]

sns.heatmap(metricasrf_calor,annot=True)
plt.title("Mapa de Calor - Métricas do RandomForest por pergunta")
plt.show()

In [40]:
dfimportance = pd.DataFrame({
    "Feature":x.columns,
    "Importância":rfmodel.feature_importances_
}) #10 valores de variáveis mais importantes para o aprendizado de máquina em questão

In [ ]:
dfimportance.sort_values("Importância",ascending=False).head(10) #Valores estão em dummies ainda

In [42]:
dfimportance["Original"] = (dfimportance["Feature"].str.split("_").str[0]) #Agora vamos tirar os dummies

In [43]:
importance_nodummies = dfimportance.groupby("Original")["Importância"].sum().reset_index().sort_values("Importância",ascending=False)

In [ ]:
importance_nodummies #Variáveis mais importantes

In [ ]:
llmxrf = pd.DataFrame({
    "Pergunta":dfrfmetricas["Pergunta"],
    "LLM":dfmetricas["Acurácia"],
    "RF":dfrfmetricas["Acurácia"]
})

longlxr = llmxrf.melt(id_vars="Pergunta",value_vars=['LLM','RF'],var_name="Modelo",value_name="Acurácia")
sns.barplot(x="Pergunta",y="Acurácia",data=longlxr,errorbar=None,hue="Modelo")
plt.title("Gráfico Final - Acurácia do LLM vs RandomForest")
plt.show()